In [1]:
import pandas as pd
import glob
import gc  # for memory clearup

In [2]:
!pwd

/home/jovyan/work/code


In [5]:
input_path = "../data/Cell/NB/"

files = glob.glob(input_path + "*.csv")
print(files)  # use the string slicing method to get the T.. information?

tumor_list = [file_name.split("_")[-5] for file_name in files]
print(tumor_list)

['../data/Cell/NB/GSM4654669_T162_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4088782_T75_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4654672_T200_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4088776_T27_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4088778_T40_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4654674_T230_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4654673_T214_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4088779_T44_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4088783_T92_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4654671_T188_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4088777_T34_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4088781_T71_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4088780_T69_gene_cell_exprs_table.csv', '../data/Cell/NB/GSM4654670_T175_gene_cell_exprs_table.csv']
['T162', 'T75', 'T200', 'T27', 'T40', 'T230', 'T214', 'T44', 'T92', 'T188', 'T34', 'T71', 'T69', 'T175']


In [6]:
pd_dataframe_list = []
for file in files:
    df = pd.read_csv(file)
    sliced = df.loc[:, df.columns!="Gene_ID"]
    pd_dataframe_list.append(sliced)
    

del files
gc.collect()

0

after deleting "Gene_ID" from some files, prefix all columns' names with tumor_id_ except the "Symbol" column

In [7]:
# tumor_list = ["T27", "T34", "T40", "T44", "T69", "T71", "T75", "T92", "T162", "T174", "T188", "T200", "T214", "T230", "F2", "F7", "F106", "F107"]  # it also includes fetal adrenal glands
# the order of tumor_list needs to consider the file order in the folder
for i in range(len(tumor_list)):
    pd_dataframe_list[i] = pd_dataframe_list[i].add_prefix(tumor_list[i]+"_")
    pd_dataframe_list[i].rename(columns={tumor_list[i]+"_Symbol": "Symbol"}, inplace=True)
    
for df in pd_dataframe_list:
    print(df.columns)

Index(['Symbol', 'T162_AAACCCAAGACCATTC.1', 'T162_AAACCCAAGATCCAAA.1',
       'T162_AAACCCAAGATGAAGG.1', 'T162_AAACCCAAGCAAACAT.1',
       'T162_AAACCCAAGCACAAAT.1', 'T162_AAACCCAAGCACTTTG.1',
       'T162_AAACCCAAGCTCCACG.1', 'T162_AAACCCAAGGCAGGGA.1',
       'T162_AAACCCAAGGCATGGT.1',
       ...
       'T162_GCCGATGTCGTTCCTG.1', 'T162_GCCGATGTCTAGACAC.1',
       'T162_GCCGATGTCTCGACCT.1', 'T162_GCCGTGAAGAAGGCTC.1',
       'T162_GCCGTGAAGCCAACCC.1', 'T162_GCCGTGAAGGTAATCA.1',
       'T162_GCCGTGAAGTGGACTG.1', 'T162_GCCGTGACAAGCGCAA.1',
       'T162_GCCGTGACACCCTGAG.1', 'T162_GCCGTGACACGAAGAC.1'],
      dtype='object', length=16385)
Index(['Symbol', 'T75_AAACCTGAGATCCCAT.1', 'T75_AAACCTGAGCCTTGAT.1',
       'T75_AAACCTGAGCGATAGC.1', 'T75_AAACCTGAGCGTTTAC.1',
       'T75_AAACCTGAGGAATGGA.1', 'T75_AAACCTGAGGAGTCTG.1',
       'T75_AAACCTGAGTACTTGC.1', 'T75_AAACCTGCAACTGCTA.1',
       'T75_AAACCTGCAAGAGGCT.1',
       ...
       'T75_TTTGGTTAGGACACCA.1', 'T75_TTTGGTTAGGAGTTTA.1',
       'T7

output the renamed files

In [8]:
tumor_list[0]

'T162'

In [9]:
output_path = "../data/Cell/NB_renamed/"
for i in range(len(tumor_list)):
    pd_dataframe_list[i].to_csv(output_path + tumor_list[i] + ".csv", index=False)
    

del pd_dataframe_list
gc.collect()

0

import the renamed files and then horizontally concatenate them by Symbol

In [10]:
input_path = "../data/Cell/NB_renamed/"

files = glob.glob(input_path + "*.csv")
concatenate_list = [pd.read_csv(file, index_col=0) for file in files]
for file in concatenate_list:
    # print(file.head())
    duplicated_list = file[file.index.duplicated(keep=False)].index.to_list()
    duplicated_list = list(set(duplicated_list))  # remove duplicates
    for duplicated_symbol in duplicated_list:
        insert = file.loc[duplicated_symbol].mean()  # calculate the mean expression of duplicated symbols for each cell
        file.drop(duplicated_symbol, axis="index", inplace=True)  # delete the duplicated symbols first
        file.loc[duplicated_symbol] = insert  # insert back the mean
    print(file[file.index.duplicated()])  # check is there duplicated symbols?

Empty DataFrame
Columns: [T44_AAACCTGAGAGGTTGC.1, T44_AAACCTGAGAGTGAGA.1, T44_AAACCTGAGGCATGTG.1, T44_AAACCTGAGTGTACCT.1, T44_AAACCTGCAATGCCAT.1, T44_AAACCTGCAATGGATA.1, T44_AAACCTGCACTGTGTA.1, T44_AAACCTGGTAATCACC.1, T44_AAACCTGGTAGTGAAT.1, T44_AAACCTGGTTGGACCC.1, T44_AAACCTGGTTGTACAC.1, T44_AAACCTGTCAAACGGG.1, T44_AAACCTGTCCGTAGTA.1, T44_AAACGGGAGCCACGCT.1, T44_AAACGGGAGCCCAATT.1, T44_AAACGGGAGTCATCCA.1, T44_AAACGGGAGTGGTAGC.1, T44_AAACGGGCAGATCTGT.1, T44_AAACGGGGTAAATGTG.1, T44_AAACGGGGTAGCGCAA.1, T44_AAACGGGGTGAAATCA.1, T44_AAACGGGTCACTTACT.1, T44_AAACGGGTCCACTGGG.1, T44_AAACGGGTCGAGCCCA.1, T44_AAACGGGTCGTCTGAA.1, T44_AAAGATGAGAGGGCTT.1, T44_AAAGATGAGGATGCGT.1, T44_AAAGATGTCCGCGTTT.1, T44_AAAGATGTCCTTTACA.1, T44_AAAGATGTCGCTTAGA.1, T44_AAAGATGTCTCAACTT.1, T44_AAAGCAAAGCCAGAAC.1, T44_AAAGCAAAGCCAGTTT.1, T44_AAAGCAAAGTGACATA.1, T44_AAAGCAACACTGTGTA.1, T44_AAAGCAACAGGATTGG.1, T44_AAAGCAAGTCGCCATG.1, T44_AAAGCAAGTCGGGTCT.1, T44_AAAGCAAGTTTCGCTC.1, T44_AAAGCAATCATAAAGG.1, T44_AAAGTAGAGA

In [11]:
horizontally_concatenated_csv_file = pd.concat(concatenate_list, axis=1, join="inner")
horizontally_concatenated_csv_file.head()


del files
del concatenate_list
del duplicated_list
gc.collect()

0

In [12]:
output_path = "../data/Cell/"
# save file
horizontally_concatenated_csv_file.to_csv(output_path + "concatenated.csv")

transpose the concatenated file

In [13]:
# horizontally_concatenated_csv_file = pd.read_csv("/hdd/edward/the-project/data/GSE137804_RAW/Patient-derived-biopsy-tissues/csv-concatenated/concatenated.csv")
# print(horizontally_concatenated_csv_file.head())
transposed = horizontally_concatenated_csv_file.T
transposed.head()

Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,MATR3,RGS5
T44_AAACCTGAGAGGTTGC.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0
T44_AAACCTGAGAGTGAGA.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,3.5
T44_AAACCTGAGGCATGTG.1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,4.0
T44_AAACCTGAGTGTACCT.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,5.5
T44_AAACCTGCAATGCCAT.1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0


In [14]:
output_path = "../data/Cell/"

print(transposed.columns[:5])
print(transposed.index[:5])
transposed.to_csv(output_path + "transposed.csv")


del horizontally_concatenated_csv_file
gc.collect()

Index(['TSPAN6', 'DPM1', 'SCYL3', 'C1orf112', 'FGR'], dtype='object', name='Symbol')
Index(['T44_AAACCTGAGAGGTTGC.1', 'T44_AAACCTGAGAGTGAGA.1',
       'T44_AAACCTGAGGCATGTG.1', 'T44_AAACCTGAGTGTACCT.1',
       'T44_AAACCTGCAATGCCAT.1'],
      dtype='object')


8

check whether the data is good, remove duplicates, NA

In [15]:
transposed[transposed.index.duplicated()]

Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,MATR3,RGS5


In [16]:
transposed.T[transposed.T.index.duplicated()]

,T44_AAACCTGAGAGGTTGC.1,T44_AAACCTGAGAGTGAGA.1,T44_AAACCTGAGGCATGTG.1,T44_AAACCTGAGTGTACCT.1,T44_AAACCTGCAATGCCAT.1,T44_AAACCTGCAATGGATA.1,T44_AAACCTGCACTGTGTA.1,T44_AAACCTGGTAATCACC.1,T44_AAACCTGGTAGTGAAT.1,T44_AAACCTGGTTGGACCC.1,...,T40_TTTGTCACAGACAAGC.1,T40_TTTGTCACATGTTCCC.1,T40_TTTGTCAGTCATTAGC.1,T40_TTTGTCAGTCGATTGT.1,T40_TTTGTCAGTGAGCGAT.1,T40_TTTGTCAGTGTCCTCT.1,T40_TTTGTCATCACTTATC.1,T40_TTTGTCATCAGGCCCA.1,T40_TTTGTCATCTGCAGTA.1,T40_TTTGTCATCTGGAGCC.1
Symbol,,,,,,,,,,,,,,,,,,,,,


In [17]:
set(transposed.dtypes)

{dtype('float64')}

In [18]:
transposed.isna().sum()

Symbol
TSPAN6       6
DPM1         6
SCYL3        6
C1orf112     6
FGR          6
            ..
LINC01481    6
CYB561D2     6
CCDC7        6
MATR3        6
RGS5         6
Length: 12939, dtype: int64

In [19]:
transposed.isnull().any()

Symbol
TSPAN6       True
DPM1         True
SCYL3        True
C1orf112     True
FGR          True
             ... 
LINC01481    True
CYB561D2     True
CCDC7        True
MATR3        True
RGS5         True
Length: 12939, dtype: bool

In [20]:
transposed.isnull().values.any()

True

In [21]:
transposed.dropna(inplace=True)
transposed

Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,MATR3,RGS5
T44_AAACCTGAGAGGTTGC.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0
T44_AAACCTGAGAGTGAGA.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,3.5
T44_AAACCTGAGGCATGTG.1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,4.0
T44_AAACCTGAGTGTACCT.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,5.5
T44_AAACCTGCAATGCCAT.1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T40_TTTGTCAGTGTCCTCT.1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.5
T40_TTTGTCATCACTTATC.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,16.0
T40_TTTGTCATCAGGCCCA.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
T40_TTTGTCATCTGCAGTA.1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0


In [22]:
transposed.isnull().values.any()

False

In [23]:
output_path = "../data/Cell/"

print(transposed.columns[:5])
print(transposed.index[:5])
transposed.to_csv(output_path + "cleaned.csv")


del transposed
gc.collect()

Index(['TSPAN6', 'DPM1', 'SCYL3', 'C1orf112', 'FGR'], dtype='object', name='Symbol')
Index(['T44_AAACCTGAGAGGTTGC.1', 'T44_AAACCTGAGAGTGAGA.1',
       'T44_AAACCTGAGGCATGTG.1', 'T44_AAACCTGAGTGTACCT.1',
       'T44_AAACCTGCAATGCCAT.1'],
      dtype='object')


10

transfer to anndata

In [24]:
import anndata as ad
import pandas as pd

pre_anndata = pd.read_csv("../data/Cell/cleaned.csv", index_col=0)
# pre_anndata = transposed
pre_anndata

,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,MATR3,RGS5
T44_AAACCTGAGAGGTTGC.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0
T44_AAACCTGAGAGTGAGA.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,3.5
T44_AAACCTGAGGCATGTG.1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,4.0
T44_AAACCTGAGTGTACCT.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,5.5
T44_AAACCTGCAATGCCAT.1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T40_TTTGTCAGTGTCCTCT.1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.5
T40_TTTGTCATCACTTATC.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,16.0
T40_TTTGTCATCAGGCCCA.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
T40_TTTGTCATCTGCAGTA.1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0


In [25]:
# pre_anndata.set_index("Symbol", inplace=True)
pre_anndata.rename_axis("Cell_barcode", inplace=True)
pre_anndata.rename_axis("Symbol", axis="columns", inplace=True)
pre_anndata.head()

Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,MATR3,RGS5
Cell_barcode,,,,,,,,,,,,,,,,,,,,,
T44_AAACCTGAGAGGTTGC.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0
T44_AAACCTGAGAGTGAGA.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,3.5
T44_AAACCTGAGGCATGTG.1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,4.0
T44_AAACCTGAGTGTACCT.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,5.5
T44_AAACCTGCAATGCCAT.1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0


output the anndata object to a hdf5 file

In [26]:
adata = ad.AnnData(pre_anndata)
adata

del pre_anndata
gc.collect()

0

In [27]:
output_path = "../data/Cell/"
adata.write(output_path + "counts.h5ad")